# 01 — Harmonização dos dados TIC Domicílios (2023-2025)

Este notebook carrega os 6 microdados originais, harmoniza os nomes de colunas entre os 3 anos, recodifica as variáveis principais e salva o resultado em `data/processed/`.

Ver `docs/documentacao_projeto.md` para o detalhamento de cada decisão de harmonização, e `docs/caderno_de_bordo.md` para o histórico de decisões (inclusive a descoberta dos dois sistemas de peso, seção abaixo).

## 1. Carregar os dados brutos

Atenção: os CSVs usam separador `;` e decimal `,` (formato brasileiro) — sem isso, o `PESO` é lido como texto.

In [14]:
import pandas as pd
import sys
import os

def encontrar_raiz_projeto(pasta_atual):
    """Sobe diretorios ate achar a pasta que contem src/harmonizacao.py.

    Evita depender de quantos niveis subir (sys.path.append('..') quebra se o
    kernel Jupyter nao estiver rodando de dentro de notebooks/, o que e comum
    no VS Code -- ele costuma rodar da raiz do repositorio).
    """
    pasta = os.path.abspath(pasta_atual)
    while True:
        if os.path.exists(os.path.join(pasta, 'src', 'harmonizacao.py')):
            return pasta
        pasta_pai = os.path.dirname(pasta)
        if pasta_pai == pasta:
            raise FileNotFoundError('Nao encontrei src/harmonizacao.py subindo a partir de ' + pasta_atual)
        pasta = pasta_pai

raiz_projeto = encontrar_raiz_projeto(os.getcwd())
if raiz_projeto not in sys.path:
    sys.path.append(raiz_projeto)

from src.harmonizacao import recodificar_g2, recodificar_velocidade, renomear_colunas, padronizar_pesos_replica

ind_2023 = pd.read_csv(os.path.join(raiz_projeto, 'data/raw/individuos/2023_individuos.csv'), sep=';', decimal=',')
ind_2024 = pd.read_csv(os.path.join(raiz_projeto, 'data/raw/individuos/2024_individuos.csv'), sep=';', decimal=',')
ind_2025 = pd.read_csv(os.path.join(raiz_projeto, 'data/raw/individuos/2025_individuos.csv'), sep=';', decimal=',')

dom_2023 = pd.read_csv(os.path.join(raiz_projeto, 'data/raw/domicilios/2023_domicilios.csv'), sep=';', decimal=',')
dom_2024 = pd.read_csv(os.path.join(raiz_projeto, 'data/raw/domicilios/2024_domicilios.csv'), sep=';', decimal=',')
dom_2025 = pd.read_csv(os.path.join(raiz_projeto, 'data/raw/domicilios/2025_domicilios.csv'), sep=';', decimal=',')


## 2. Dois sistemas de peso amostral — achado importante

O módulo `domicilios` e o módulo `individuos` **cada um tem seu próprio `PESO`**, e não são a mesma coisa: testamos em 2023 e 100% das 21.271 pessoas têm um peso de indivíduo diferente do peso da própria casa. As 200 colunas de peso réplica (`rep1`...`rep200`, usadas no Rao-Scott) também divergem entre os módulos (13.798 de 21.271 pessoas diferem só no `rep1`).

Isso faz sentido: a chance de uma *casa* ser sorteada na amostra é diferente da chance de uma *pessoa específica* dentro da casa ser sorteada. Como a bateria G1/G2 (o coração da pergunta de pesquisa) é respondida por pessoa, **o peso certo pra qualquer teste envolvendo G1/G2 é o peso do indivíduo, não o do domicílio** — mesmo quando cruzado com variável de domicílio tipo renda ou região, porque a unidade de análise continua sendo a pessoa.

Por isso o crosswalk agora tem `peso_amostral_domicilio` e `peso_amostral_individuo` como variáveis separadas, nunca uma genérica `peso_amostral` que poderia ser usada errado sem avisar.

## 3. Harmonizar nomes de colunas entre os anos

Usando o crosswalk (`docs/lista_final_variaveis_projeto.csv`).

In [15]:
crosswalk = pd.read_csv(os.path.join(raiz_projeto, 'docs/lista_final_variaveis_projeto.csv'))
crosswalk

,variavel_final,modulo,nome_2023,nome_2024,nome_2025,precisa_recodificar,observacao
0,chave_domicilio,domicilios+individuos,ID_DOMICILIO,ID_DOMICILIO,ID_DOMICILIO,nao,chave de juncao entre os modulos domicilios e ...
1,peso_amostral_domicilio,domicilios,PESO,PESO,PESO,nao,peso amostral do DOMICILIO; usar so em estatis...
2,peso_amostral_individuo,individuos,PESO,PESO,PESO,nao,peso amostral do RESPONDENTE INDIVIDUAL (16+);...
3,renda_familiar,domicilios,RENDA_FAMILIAR_2,RENDA_FAMILIAR_2,RENDA_FAMILIAR_2,nao,6 faixas de salario minimo idênticas; NAO usar...
4,classe_economica,domicilios,CLASSE_2015,CLASSE_2015,CLASSE_2015,nao,Criterio Brasil 2015 identica nos 3 anos
5,escolaridade,individuos,GRAU_INSTRUCAO_2,GRAU_INST_1,GRAU_INST_1,sim (rename),versao agregada 4 categorias identicas; NAO us...
6,raca,individuos,RACA,RACA,RACA,nao,5 categorias validas (1=Branca 2=Preta 3=Parda...
7,sexo,individuos,SEXO,SEXO,SEXO,nao,identica nos 3 anos
8,idade,individuos,IDADE,IDADE,IDADE,nao,identica nos 3 anos
9,regiao,domicilios,COD_REGIAO_2,COD_REGIAO_2,COD_REGIAO_2,nao,identica nos 3 anos


### 3.1 Padronizar os pesos réplica antes de renomear

Achado da primeira sessão: `rep1`...`rep200` vêm minúsculo em domicílios-2023, maiúsculo (`REP1`...`REP200`) em domicílios-2024/2025. O módulo indivíduos já vem sempre minúsculo nos 3 anos — então aplicar a mesma função nele não muda nada, é só por consistência/segurança.

In [16]:
dom_2023 = padronizar_pesos_replica(dom_2023)
dom_2024 = padronizar_pesos_replica(dom_2024)
dom_2025 = padronizar_pesos_replica(dom_2025)

ind_2023 = padronizar_pesos_replica(ind_2023)
ind_2024 = padronizar_pesos_replica(ind_2024)
ind_2025 = padronizar_pesos_replica(ind_2025)

### 3.2 Aplicar o rename

In [17]:
dom_2023 = renomear_colunas(dom_2023, 2023, 'domicilios', crosswalk)
dom_2024 = renomear_colunas(dom_2024, 2024, 'domicilios', crosswalk)
dom_2025 = renomear_colunas(dom_2025, 2025, 'domicilios', crosswalk)

ind_2023 = renomear_colunas(ind_2023, 2023, 'individuos', crosswalk)
ind_2024 = renomear_colunas(ind_2024, 2024, 'individuos', crosswalk)
ind_2025 = renomear_colunas(ind_2025, 2025, 'individuos', crosswalk)

## 4. Marcar o ano de origem

Precisamos saber, depois de juntar os 3 anos, de qual ano cada linha veio.

In [18]:
for df, ano in [(dom_2023, 2023), (dom_2024, 2024), (dom_2025, 2025),
                (ind_2023, 2023), (ind_2024, 2024), (ind_2025, 2025)]:
    df['ano'] = ano

C:\Users\pedro\AppData\Local\Temp\ipykernel_12892\3322738575.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ano'] = ano


## 5. Recodificar G1/G2 (bateria de uso funcional)

G2 vira binário (1=conseguiu, 0=não conseguiu, NaN=missing). G1 já vem em 0/1, só precisa tratar os códigos de missing (97/98/99) como `NaN`. A área H (Justiça) só existe em 2025.

In [19]:
for df, ano in [(ind_2023, 2023), (ind_2024, 2024), (ind_2025, 2025)]:
    areas = ['A', 'B', 'C', 'D', 'E', 'F', 'G'] + (['H'] if ano == 2025 else [])
    for area in areas:
        df[f'G2_{area}'] = recodificar_g2(df[f'G2_{area}'])
        df[f'G1_{area}'] = df[f'G1_{area}'].replace({97: None, 98: None, 99: None})

## 5.1 Tratar código de missing em `raca`

**Achado (21/09), confirmado (21/09):** a variável `raca` tem um 6º valor, `98`, além das 5 categorias documentadas (2.285 pessoas, ~3,4% da base, nos 3 anos). **Confirmado no dicionário oficial** (`docs/fontes_cetic/dicionarios/individuos/`, idêntico em 2023/2024/2025): `98 = "Não respondeu"`. Recodificado pra `NaN`.

In [20]:
# Ver ressalva na celula acima: tratamento provisorio, confirmar no dicionario oficial
for df in [ind_2023, ind_2024, ind_2025]:
    df['raca'] = df['raca'].replace({98: None})

## 6. Recodificar velocidade de conexão

A coluna já foi renomeada pra `velocidade_conexao`, mas ainda guarda o código numérico bruto (1 a 10 em 2023/2024, 1 a 7 em 2025). Agora aplicamos a função de fato.

In [21]:
dom_2023['velocidade_conexao'] = recodificar_velocidade(dom_2023['velocidade_conexao'], 2023)
dom_2024['velocidade_conexao'] = recodificar_velocidade(dom_2024['velocidade_conexao'], 2024)
dom_2025['velocidade_conexao'] = recodificar_velocidade(dom_2025['velocidade_conexao'], 2025)

## 7. Selecionar colunas de interesse

Os pesos réplica (200 colunas) são pesados: incluí-los direto no arquivo principal deixou os CSVs enormes (testamos: 92 a 137 MB cada, acima do limite prático do GitHub). Como eles só servem pra uma etapa específica lá na frente (o teste Rao-Scott, notebook 03), fica melhor separá-los num arquivo próprio, carregado só quando for a hora de rodar o teste.

`domicilios_harmonizado` e `individuos_harmonizado` ficam enxutos, com o peso principal (não as réplicas). Só o módulo indivíduos leva réplicas pra um arquivo dedicado — é o conjunto certo pra qualquer teste envolvendo G1/G2 (ver Seção 2), então não faz sentido gastar espaço com as réplicas de domicílio, que não vamos usar.

In [22]:
cols_dom = ['chave_domicilio', 'ano', 'peso_amostral_domicilio', 'renda_familiar', 'classe_economica',
            'regiao', 'area_urbano_rural', 'velocidade_conexao', 'valor_conexao']

cols_dom_para_merge = ['chave_domicilio', 'ano', 'renda_familiar', 'classe_economica',
                       'regiao', 'area_urbano_rural', 'velocidade_conexao', 'valor_conexao']

cols_ind_fixas = ['chave_domicilio', 'ID_MORADOR', 'ano', 'idade', 'sexo', 'raca', 'escolaridade',
                  'g1_agreg', 'transacoes_financeiras', 'peso_amostral_individuo']

cols_replica_individuo = ['chave_domicilio', 'ID_MORADOR', 'ano', 'peso_amostral_individuo'] + \
                          [f'rep{i}' for i in range(1, 201)]

## 8. Concatenar os 3 anos e juntar os módulos

- `pd.concat`: empilha os 3 anos de cada módulo
- `pd.merge`: junta indivíduos com os atributos do domicílio (renda, região...), usando `chave_domicilio` + `ano`. `cols_dom_para_merge` não inclui peso de domicílio, de propósito, pra `dados_harmonizados` nunca ficar com dois sistemas de peso ambíguos misturados.

`individuos_pesos_replica` é o arquivo à parte com as 200 réplicas, salvo pra ser usado só no notebook de testes estatísticos.

In [23]:
domicilios_harmonizado = pd.concat(
    [dom_2023[cols_dom], dom_2024[cols_dom], dom_2025[cols_dom]],
    ignore_index=True
)

partes_individuos = []
partes_replica = []
for df in [ind_2023, ind_2024, ind_2025]:
    colunas_g1_g2 = [c for c in df.columns if c.startswith('G1_') or c.startswith('G2_')]
    partes_individuos.append(df[cols_ind_fixas + colunas_g1_g2])
    partes_replica.append(df[cols_replica_individuo])

individuos_harmonizado = pd.concat(partes_individuos, ignore_index=True)
individuos_pesos_replica = pd.concat(partes_replica, ignore_index=True)

dados_harmonizados = individuos_harmonizado.merge(
    domicilios_harmonizado[cols_dom_para_merge], on=['chave_domicilio', 'ano'], how='left'
)

## 9. Conferir o resultado

Checagens antes de salvar: (1) o número de linhas bate com a soma dos 3 anos originais, (2) G2_A só tem 0, 1 ou NaN, (3) velocidade_conexao só tem as 4 categorias esperadas, (4) os dois pesos continuam diferentes um do outro (prova de que não veio da mesma coluna por engano).

In [24]:
print('domicilios_harmonizado:', domicilios_harmonizado.shape)
print('individuos_harmonizado:', individuos_harmonizado.shape)
print('dados_harmonizados (merge):', dados_harmonizados.shape)
print()
print('G2_A valores unicos:', sorted(dados_harmonizados['G2_A'].dropna().unique()))
print('velocidade_conexao valores unicos:', dados_harmonizados['velocidade_conexao'].unique())
print()
print('peso_amostral_individuo existe em dados_harmonizados?',
      'peso_amostral_individuo' in dados_harmonizados.columns)
print('peso_amostral_domicilio NAO deveria estar em dados_harmonizados:',
      'peso_amostral_domicilio' not in dados_harmonizados.columns)

domicilios_harmonizado: (75008, 9)
individuos_harmonizado: (66976, 26)
dados_harmonizados (merge): (66976, 32)

G2_A valores unicos: [np.float64(0.0), np.float64(1.0)]
velocidade_conexao valores unicos: <StringArray>
[nan, 'baixa', 'media', 'alta', 'sem conexão']
Length: 5, dtype: str

peso_amostral_individuo existe em dados_harmonizados? True
peso_amostral_domicilio NAO deveria estar em dados_harmonizados: True


## 10. Salvar em data/processed/

`individuos_pesos_replica.csv` fica pesado (200 colunas numéricas) — vale considerar deixar ele fora do Git (igual já se faz com `data/raw/`), documentando no README que ele é regenerado rodando este notebook, em vez de versionado.

In [25]:
domicilios_harmonizado.to_csv(os.path.join(raiz_projeto, 'data/processed/domicilios_harmonizado.csv'), index=False)
individuos_harmonizado.to_csv(os.path.join(raiz_projeto, 'data/processed/individuos_harmonizado.csv'), index=False)
dados_harmonizados.to_csv(os.path.join(raiz_projeto, 'data/processed/dados_harmonizados.csv'), index=False)
individuos_pesos_replica.to_csv(os.path.join(raiz_projeto, 'data/processed/individuos_pesos_replica.csv'), index=False)